# Clean up ShareChat

- Dataset - https://huggingface.co/datasets/tucnguyen/ShareChat/tree/main
- Paper - https://arxiv.org/abs/2512.17843

In [1]:
import pandas as pd

In [3]:
# takes a few min
# Let's get: 1st turn only; English language; strings <= 131 characters (same filtering as WildChat)

dfs = []
models = ['claude', 'grok', 'perplexity', 'gemini', 'chatgpt']

for model in models:
    dfs.append(
        pd.read_csv(
            f'sharechat/{model}_results_final_language_filtered.csv'
        ).query(
            'detected_language_final == "English"'
        ).query(
            'role == "user"'
        ).query(
            'message_index == 1' # Let's look at the first message from the user only
        )
        .assign(
            conv_id=lambda df_: 'sharechat::' + df_.url,
            first_turn=lambda df_: df_.plain_text.str.strip(),
            first_turn_len=lambda df_: df_.first_turn.str.len(),
        )
    )
    
data_en = pd.concat(dfs)

data = (data_en
    .query('first_turn_len <= 131')
    .drop_duplicates(subset=['first_turn', 'platform'])
)

In [5]:
data_en.first_turn_len.le(131).mean().round(3)

0.623

In [20]:
data.filter(['conv_id', 'first_turn']).to_csv('interim/sharechat.first_turn.en.len131.csv.zip')